# P03. Calidad y consulta diferida

10 de octubre de 2026, sesión 4

Preparado por el PhD Esteban Hernández, CyberColombia.

Estos ejercicios usan tres muestras reales de 1.000 filas. Las primeras filas sirven para probar código, sin representatividad estadística. Los archivos originales permanecen intactos. No se requiere GPU ni una cuenta externa. Instalar requirements.txt y requirements_pqrs.txt y seleccionar el kernel del mismo entorno.


**Entorno:** Ubuntu-26.04 sobre WSL. Clonar siempre `main` en `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. Activar `.venv` desde la raíz y abrir Jupyter desde `kit/`. Seguir [la guía WSL](../../docs/instalacion-wsl.md).

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import sys, json
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "kit", Path.cwd() / "big-data-postgraduate" / "kit"]
KIT = next((p.resolve() for p in candidates if (p / "pqrs_talleres.py").exists()), None)
assert KIT is not None, "Abrir desde bigdata/, la raíz del repositorio, kit/ o kit/Notebooks/"
sys.path.insert(0, str(KIT))
import pqrs_talleres as lab
a = SimpleNamespace(datos=None, completo=False, bloque=500, out=KIT / "salidas" / "notebooks", repeticiones=3)
a.out.mkdir(parents=True, exist_ok=True)


In [ ]:
lab.formatos(a)
control = lab.calidad(a)
assert control["filas"] == control["claves_candidatas"] == 3000
assert control["exclusiones_automaticas"] == 0
control

In [ ]:
import polars as pl
q = (pl.scan_parquet(a.out / "pqrs.parquet")
     .filter(pl.col("anio") == 2024)
     .group_by("corte_archivo", "mes_num")
     .agg(pl.len().alias("reportes")))
print(q.explain())
r = q.collect(engine="streaming").sort("corte_archivo", "mes_num")
assert r["reportes"].sum() == 2000
r

## Entrega
Tabla de controles con evidencia y decisión de publicar, revisar o detener. Las 41 edades mayores de 90 se conservan. Revisar los códigos sin correspondencia DIVIPOLA y la ubicación del peticionario por separado. Un archivo de reportes no permite estimar prevalencia o riesgo clínico.

In [ ]:
import duckdb
import pandas as pd
con = duckdb.connect()
sql = con.execute("SELECT corte_archivo, mes_num, count(*) AS reportes FROM read_parquet(?) WHERE anio = 2024 GROUP BY corte_archivo, mes_num ORDER BY corte_archivo, mes_num", [str(a.out / "pqrs.parquet")]).df()
con.close()
pd.testing.assert_frame_equal(r.to_pandas().reset_index(drop=True), sql.reset_index(drop=True), check_dtype=False)
assert int(sql["reportes"].sum()) == 2000
sql